# Qwen-2.5-3B + GOR: So Sánh Trực Tiếp AdamW vs GEO-Muon
Notebook này chạy huấn luyện đối chứng có kiểm soát trên **Qwen-2.5-3B (3.09B, 36 tầng, 72 trạm GOR)** trên tập dữ liệu **Suy luận Logic (`GSM8K`, `ARC-Challenge`) + Lập trình (`Python-Alpaca`, `MBPP`)**:

0. `Qwen-2.5-3B (Base Untouched)` — Mốc gốc chưa chỉnh sửa (Zero-shot trên tập test Logic + Coding, chạy ~15 giây)
1. `Qwen-2.5-3B + GOR (AdamW)` — Gắn GOR vào đủ 72 trạm (Attention + MLP) + tối ưu `AdamW`
2. `Qwen-2.5-3B + GOR (GEO-Muon)` — Gắn GOR vào đủ 72 trạm (Attention + MLP) + tối ưu trực giao cực `GEO-Muon`

Cả 2 cấu hình huấn luyện dùng **chung đúng 1 bộ dữ liệu train (4.000 mẫu: 50% Logic + 50% Coding)**, **chung thứ tự batch (`seed=42`)**, và được đánh giá trên **cùng 200 mẫu test tách biệt hoàn toàn** (100 Logic + 100 Coding) bằng bộ 6 chỉ số cơ học.

## 1. Kiểm Tra GPU A100 & Cài Thư Viện

In [ ]:
import os
os.system("nvidia-smi")
os.system("pip install -q transformers accelerate datasets matplotlib torch")
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("BF16 Supported:", torch.cuda.is_bf16_supported())

## 2. Đồng Bộ Mã Nguồn `qwen_upcycling` Mới Nhất Từ GitHub

In [ ]:
import os
import sys
if not os.path.exists("Alba-model"):
    os.system("git clone https://github.com/potatips/Alba-model.git")
else:
    os.system("cd Alba-model && git pull origin main")
work_dir = os.path.abspath("Alba-model/qwen_upcycling")
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)
print("Loaded modules from:", work_dir)

## 3. Chuẩn Bị Bộ Dữ Liệu Suy Luận Logic & Lập Trình (4.000 Train / 200 Held-Out Eval)
- **50% Suy luận Logic & Toán (`2.000` train / `100` eval):** `openai/gsm8k` (70%) + `allenai/ai2_arc ARC-Challenge` (30%).
- **50% Coding (`2.000` train / `100` eval):** `iamtarun/python_code_instructions_18k_alpaca` + `google-research-datasets/mbpp (sanitized)`.

In [ ]:
from qwen_dataset import build_or_load_logic_code_dataset

DATASET_CACHE = "qwen_logic_code_dataset.json"
dataset_payload = build_or_load_logic_code_dataset(
    cache_path=DATASET_CACHE,
    n_train_logic=2000,
    n_train_code=2000,
    n_eval_logic=100,
    n_eval_code=100,
    seed=42,
)
print("Dataset Summary:", dataset_payload["meta"])
print("\n--- Sample Logic ---\n", dataset_payload["eval_logic"][0]["text"][:350])
print("\n--- Sample Coding ---\n", dataset_payload["eval_code"][0]["text"][:350])

## 4. Huấn Luyện `Qwen-2.5-3B + GOR` Với `AdamW` vs `GEO-Muon`
- Gắn **GOR (`0-Param`, `k=3.0`)** vào đủ `72` trạm residual (`36` Attention + `36` MLP) trên `Qwen-2.5-3B` gốc (giữ nguyên toàn bộ `3.09B` tham số dense).
- Tự động lưu kết quả sau từng biến thể vào `qwen3b_gor_results.json` (tự động tiếp tục nếu Colab ngắt kết nối). Muốn lưu trọng số `.pt` sau khi train, đặt `save_checkpoint_dir="checkpoints_qwen3b_gor"`.

In [ ]:
from qwen_train_upcycled import GOR_ONLY_VARIANTS, run_qwen3b_ablation_suite

results = run_qwen3b_ablation_suite(
    model_name="Qwen/Qwen2.5-3B",
    total_steps=200,
    seq_len=512,
    micro_batch_size=2,
    grad_accum_steps=8,
    n_train_logic=2000,
    n_train_code=2000,
    n_eval_logic=100,
    n_eval_code=100,
    adamw_lr=5e-5,
    muon_lr=1e-4,
    dataset_cache_path="qwen_logic_code_dataset.json",
    results_cache_path="qwen3b_gor_results.json",
    output_html_path="qwen3b_gor_report.html",
    output_png_path="qwen3b_gor_grid.png",
    save_checkpoint_dir=None,
    variants=GOR_ONLY_VARIANTS,
    seed=42,
)

## 5. Xem Biểu Đồ & Tải Về Báo Cáo So Sánh (`qwen3b_gor_report.html`)

In [ ]:
from IPython.display import Image, display
if os.path.exists("qwen3b_gor_grid.png"):
    display(Image("qwen3b_gor_grid.png"))
try:
    from google.colab import files
    files.download("qwen3b_gor_report.html")
    files.download("qwen3b_gor_results.json")
    files.download("qwen3b_gor_grid.png")
except Exception:
    print("Report saved locally at qwen3b_gor_report.html")